In [118]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

In [119]:
def load_and_explore_dataset(file_path):
    print("=" * 60)
    print("LOAD AND EXPLORE DATASET")
    print("=" * 60)
    
    pd.set_option('display.float_format', lambda x: f"{x:.2f}")

    df = pd.read_csv(file_path)
    print("Shape of dataset")
    print(df.shape)
    print("\nChecking for missing value:")
    print(df.isnull().sum())
    print("\nFirst five rows:")
    print(df.head())
    print("\nDescriptive statistic:")
    print(df.describe())
    print("\nDataset info:")
    print(df.info())
    print("\nIs_Boost Distribution:")
    print(df["Is_Boost"].value_counts())
    print("\nRegion Distribution:")
    print(df["Region_Parent_Name"].value_counts())
    print("\nBedrooms Distribution:")
    print(df["Bedrooms"].value_counts())
    print("\nFurnishing Distribution:")
    print(df["Furnishing"].value_counts())
    print("\nBathrooms Distribution:")
    print(df["Bathrooms"].value_counts())
    print("\nProperty Size Distribution:")
    print(df["Property_Size"].value_counts())

    return df

In [120]:
def preprocessed_data(df):
    print("=" * 60)
    print("PREPROCESSING DATASET")
    print("=" * 60)

    df_preprocessed = df.copy()
    df_preprocessed = df.dropna()

    label_encoder = {}
    df_columns = ['Furnishing', 'Region_Parent_Name', 'Is_Boost']
    for col in df_columns:
        le = LabelEncoder()
        df_preprocessed[col + "_encoded"] = le.fit_transform(df_preprocessed[col])
        label_encoder[col] = le 
        print(f"\n{col} encoded")
        for i, label in enumerate(le.classes_):
            print(f"{label}: {i}")
    print("preprocessed data shape:", df_preprocessed.shape)

    return df_preprocessed, label_encoder

In [121]:
def featured_data(df_preprocessed):
    print("\n" + "=" * 60)
    print("FEATURE DATASET")
    print("=" * 60)

    feature_columns = ['Property_Size', 'Bedrooms', 'Bathrooms', 'Furnishing_encoded', 'Region_Parent_Name_encoded', 'Is_Boost_encoded']
    target = ['Price']

    X = df_preprocessed[feature_columns]
    y = df_preprocessed[target]

    print("\nFeature shape:", df_preprocessed[feature_columns].shape)
    print("\nTarget shape:" , df_preprocessed[target].shape)
    print("\nFeature columns:", feature_columns)

    return X, y, feature_columns

In [122]:
def split_data(X, y, test_size=0.2, random_state=42):
    print("\n" + "=" * 60)
    print("SPLITTING DATASET")
    print("=" * 60)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    print("\nTraining set size:", X_train.shape[0])
    print("\nTesting set size:", X_test.shape[0])

    
    print("\nTraining set range {:.2f} - {:.2f}".format(
        float(y_train.values.min()), float(y_train.values.max())
    ))
    
    print("\nTesting set range {:.2f} - {:.2f}".format(
         float(y_test.values.min()), float(y_test.values.max())
    ))
    

    return X_train, X_test, y_train, y_test

In [123]:
def scale_features(X_train, X_test):
    print("\n" + "=" * 60)
    print("SCALE FEATURES DATASET")
    print("=" * 60)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.fit_transform(X_test)

    print("\nFeatures scaled successful")
    print("\nTrain scaled features:", X_train_scaled.shape)
    print("\nTest scaled features:", X_test_scaled.shape)

    return X_train_scaled, X_test_scaled, scaler

In [124]:
def train_model(X_train_scaled, y_train, feature_columns):
    print("\n" + "=" * 60)
    print("TRAINING MODEL")
    print("=" * 60)

    model = LinearRegression()
    model.fit(X_train_scaled, y_train)

    print("\nModel Trained Successfully")
    print("\nModel Coefficient")

    for feature, coef in zip(feature_columns, model.coef_.ravel()):
        print(f"   {feature}: {coef:.2f}")
    print(f"\nModel Intercept: {float(model.intercept_[0]):.2f}")

    return model


In [125]:
def evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test):
    print("\n" + "=" * 60)
    print("EVALUAING MODEL")
    print("=" * 60)
    

    #make prediction
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    #calculate metrics
    train_r2_score = r2_score(y_train, y_train_pred)
    test_r2_score = r2_score(y_test, y_test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print("\n" + "=" * 60)


    print("MODEL PERFORMANCE")
    print("=" * 60)
    print("TRAINING SET:")
    print(f"  R2 score: {train_r2_score:.4f}")
    print(f"  RMSE: {train_rmse:.4f}")
    print(f"  MAE:  {train_mae:.4f}")

    print("\nTesting set:")
    print(f"  R2 score: {test_r2_score:.4f}")
    print(f"  RMSE: {test_rmse:.4f}")
    print(f"  MAE:  {test_mae:.4f}")

    cv_score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')

    print("\nCross validation (5 folds)")
    print(f"  R2 score: {cv_score}")
    print(f"  Cross val mean: {cv_score.mean()}")
    print(f"  Cross val std: {cv_score.std()}")

    metrics = {
        "train_r2_score": train_r2_score,
        "test_r2_score": test_r2_score,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "cv_score": cv_score

    }
        
    return metrics

In [126]:
def save_model_artifact(model, scaler, label_encoder, feature_columns):
    print("\n" + "=" * 60)
    print("SAVING MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(model, "house_price_prediction.pkl")
    print("House Price Prediction Model Saved")

    joblib.dump(scaler, "scaler_features.pkl")
    print("Scaler Features Saved")

    joblib.dump(label_encoder, "label_encoder.pkl")
    print("Label Feature Encoder Saved")

    joblib.dump(feature_columns, "feature_columns.pkl")
    print("Feature Columns Saved")

In [127]:
def rm_train_model(X_train_scaled, y_train, feature_columns):
    print("\n" + "=" * 60)
    print("RANDOM FOREST TRAINING MODEL")
    print("=" * 60)

    rm_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
    rm_model.fit(X_train_scaled, y_train)

    print("\nRandom Forest Model Trained Successfully")

    feature_importance = sorted(zip(feature_columns, rm_model.feature_importances_), 
                                key=lambda x: x[1], reverse=True)
    for feature, importance in feature_importance:
        print(f" {feature}: {float(importance)}")
 
    return rm_model

In [128]:
def rm_evaluate_model(rm_model, X_train_scaled, X_test_scaled, y_train, y_test):
    print("\n" + "=" * 60)
    print("RANDOM FOREST EVALUAING MODEL")
    print("=" * 60)
    

    #make prediction
    y_train_pred = rm_model.predict(X_train_scaled)
    y_test_pred = rm_model.predict(X_test_scaled)

    #calculate metrics

    train_r2_score = r2_score(y_train, y_train_pred)
    test_r2_score = r2_score(y_test, y_test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print("\n" + "=" * 60)


    print("RANDOM MODEL MODEL PERFORMANCE")
    print("=" * 60)
    print("TRAINING SET:")
    print(f"  R2 score: {train_r2_score:.4f}")
    print(f"  RMSE: {train_rmse:.4f}")
    print(f"  MAE:  {train_mae:.4f}")

    print("\nTesting set:")
    print(f"  R2 score: {test_r2_score:.4f}")
    print(f"  RMSE: {test_rmse:.4f}")
    print(f"  MAE:  {test_mae:.4f}")

    cv_score = cross_val_score(rm_model, X_train_scaled, y_train, cv=5, scoring='r2')

    print("\nCross validation (5 folds)")
    print(f"  R2 score: {cv_score}")
    print(f"  Cross val mean: {cv_score.mean()}")
    print(f"  Cross val std: {cv_score.std()}")

    rm_metrics = {
        "train_r2_score": train_r2_score,
        "test_r2_score": test_r2_score,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "cv_score": cv_score

    }
        
    return rm_metrics

In [129]:
def save_rm_model_artifact(rm_model):
    print("\n" + "=" * 60)
    print("SAVING RANDOM FOREST MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(rm_model, "rm_model_housing_price_prediction.pkl")
    print("Housing Price Prediction Model Saved")

In [130]:
def predict_house_price(property_size, bedrooms, bathrooms, furnishing, region_parent_name, is_boost):
    model = joblib.load("house_price_prediction.pkl")
    scaler = joblib.load("scaler_features.pkl")
    label_encoder = joblib.load("label_encoder.pkl")
    feature_columns = joblib.load("feature_columns.pkl")

    try:
        furnishing_encoded = label_encoder['Furnishing'].transform([furnishing])[0]
        region_parent_name_encoded = label_encoder['Region_Parent_Name'].transform([region_parent_name])[0]        
        is_boost_encoded = label_encoder['Is_Boost'].transform([is_boost])[0]
    except ValueError as e:
        return f"Unknown Category {e}"

    features_dict = {
        "Property_Size": property_size,
        "Bedrooms":  bedrooms,
        "Bathrooms":  bathrooms,
        "Furnishing_encoded": furnishing_encoded,
        "Region_Parent_Name_encoded": region_parent_name_encoded,
        "Is_Boost_encoded": is_boost_encoded
    }

    features = np.array([[features_dict[col] for col in features_dict]])

    feature_scale = scaler.transform(features)

    predicted_price = model.predict(feature_scale)[0].item()

    return predicted_price

In [150]:
def rm_predict_house_price(property_size, bedrooms, bathrooms, furnishing, region_parent_name, is_boost):
    rm_model = joblib.load("rm_model_housing_price_prediction.pkl")
    scaler = joblib.load("scaler_features.pkl")
    label_encoder = joblib.load("label_encoder.pkl")
    feature_columns = joblib.load("feature_columns.pkl")

    try:
        furnishing_encoded = label_encoder['Furnishing'].transform([furnishing])[0]
        region_parent_name_encoded = label_encoder['Region_Parent_Name'].transform([region_parent_name])[0]        
        is_boost_encoded = label_encoder['Is_Boost'].transform([is_boost])[0]
    except ValueError as e:
        return f"Unknown Category {e}"

    features_dict = {
        "Property_Size": property_size,
        "Bedrooms":  bedrooms,
        "Bathrooms":  bathrooms,
        "Furnishing_encoded": furnishing_encoded,
        "Region_Parent_Name_encoded": region_parent_name_encoded,
        "Is_Boost_encoded": is_boost_encoded
    }

    features = np.array([[features_dict[col] for col in features_dict]])

    feature_scale = scaler.transform(features)

    predicted_price = rm_model.predict(feature_scale)[0].item()

    return predicted_price

In [151]:
def test_prediction():
    print("\n" + "=" * 60)
    print("TEST HOUSE PREDICTION")
    print("=" * 60)

    #EXAMPLE 1
    price1 = predict_house_price(200, 2, 3,  "Furnished", "Lagos State", "Diamond")
    print("\n Furnished 2 Bedroom in Lagos (3 bathrooms, 200 sqm)")
    print(f" ₦{float(price1):,.2f}")

    #EXAMPLE 1
    price2 = predict_house_price(500, 4, 5,  "Semi-Furnished", "Abuja (FCT)", "VIP Gold")
    print("\n Semi_Furnished 4 Bedroom in Abuja (5 bathrooms, 500 sqm")
    print(f" ₦{float(price2):,.2f}")

    #EXAMPLE 1
    price3 = predict_house_price(1500, 10, 10,  "Unfurnished", "Oyo State", "Enterprise")
    print("\n Unfurnished 10 Bedroom in Oyo (10 bathrooms, 1500 sqm")
    print(f" ₦{float(price3):,.2f}")

In [152]:
def rm_test_prediction():
    print("\n" + "=" * 60)
    print("RANDOM FOREST TEST HOUSE PREDICTION")
    print("=" * 60)

    #EXAMPLE 1
    price1 = rm_predict_house_price(200, 2, 3,  "Furnished", "Lagos State", "Diamond")
    print("\n Furnished 2 Bedroom in Lagos (3 bathrooms, 200 sqm)")
    print(f" ₦{float(price1):,.2f}")

    #EXAMPLE 1
    price2 = rm_predict_house_price(500, 4, 5,  "Semi-Furnished", "Abuja (FCT)", "VIP Gold")
    print("\n Semi_Furnished 4 Bedroom in Abuja (5 bathrooms, 500 sqm")
    print(f" ₦{float(price2):,.2f}")

    #EXAMPLE 1
    price3 = rm_predict_house_price(1500, 10, 10,  "Unfurnished", "Oyo State", "Enterprise")
    print("\n Unfurnished 10 Bedroom in Oyo (10 bathrooms, 1500 sqm")
    print(f" ₦{float(price3):,.2f}")

In [153]:
def main():
    file_path = "data/jiji_housing_cleaned.csv"

    df = load_and_explore_dataset(file_path)

    df_preprocessed, label_encoder = preprocessed_data(df)
    
    X, y, feature_columns = featured_data(df_preprocessed)

    X_train, X_test, y_train, y_test = split_data(X,y)

    X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

    model = train_model(X_train_scaled, y_train, feature_columns)

    metrics = evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test)

    rm_model = rm_train_model(X_train_scaled, y_train, feature_columns)

    rm_metrics = rm_evaluate_model(rm_model, X_train_scaled, X_test_scaled, y_train, y_test)

    save_model_artifact(model, scaler, label_encoder, feature_columns)

    save_rm_model_artifact(rm_model)

    test_prediction()

    rm_test_prediction()

In [154]:
if __name__ == "__main__":
    main()

LOAD AND EXPLORE DATASET
Shape of dataset
(1753, 10)

Checking for missing value:
Title                 0
Property_Size         0
Bedrooms              0
Bathrooms             0
Furnishing            0
Region                0
Region_Name           0
Region_Parent_Name    0
Is_Boost              0
Price                 0
dtype: int64

First five rows:
                                              Title  Property_Size  Bedrooms  \
0     Furnished 3bdrm Apartment in Maitama for sale            500         3   
1        4bdrm Townhouse/Terrace in Gaduwa for sale            380         4   
2        4bdrm Townhouse/Terrace in Gaduwa for sale            300         4   
3  4bdrm House in Off Lekki-Epe Expressway for sale            500         4   
4                    5bdrm Duplex in Lekki for sale            350         5   

   Bathrooms   Furnishing                          Region  \
0          4    Furnished            Abuja (FCT), Maitama   
1          4  Unfurnished             Abuja 